# AdaBoost Benchmarking

In this notebook, we compare our custom-built AdaBoost classifier (Adaptive Boosting using Decision Stumps) against the `scikit-learn` implementation. We will evaluate how sequential weighted training affects performance on the Breast Cancer dataset.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
from time import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier as SklearnAB
from sklearn.tree import DecisionTreeClassifier

# Import our custom modules
from classical_ml.ensemble.adaboost import AdaBoost as CustomAB
from utils.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# 1. Load Dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


In [3]:
print("1. Custom AdaBoost")
start = time()

# Initialize AdaBoost with 10 estimators (decision stumps)
custom_ab = CustomAB(n_estimators=10)
custom_ab.fit(X_train, y_train)
preds_custom = custom_ab.predict(X_test)
time_custom = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_custom):.4f}")
print(f"Precision : {precision_score(y_test, preds_custom):.4f}")
print(f"Recall    : {recall_score(y_test, preds_custom):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_custom):.4f}")
print(f"Time Taken: {time_custom:.5f} seconds\n")

1. Custom AdaBoost
Accuracy  : 0.9561
Precision : 0.9583
Recall    : 0.9718
F1-Score  : 0.9650
Time Taken: 3.18420 seconds



In [ ]:
print("2. Scikit-Learn AdaBoost")
start = time()

# We set n_estimators=10
sk_ab = SklearnAB(
    estimator=DecisionTreeClassifier(max_depth=1), 
    n_estimators=10, 
    random_state=42
)
sk_ab.fit(X_train, y_train)
preds_sk = sk_ab.predict(X_test)
time_sk = time() - start

print(f"Accuracy  : {accuracy_score(y_test, preds_sk):.4f}")
print(f"Precision : {precision_score(y_test, preds_sk):.4f}")
print(f"Recall    : {recall_score(y_test, preds_sk):.4f}")
print(f"F1-Score  : {f1_score(y_test, preds_sk):.4f}")
print(f"Time Taken: {time_sk:.5f} seconds\n")

2. Scikit-Learn AdaBoost
Accuracy  : 0.9649
Precision : 0.9589
Recall    : 0.9859
F1-Score  : 0.9722
Time Taken: 0.04492 seconds



## Conclusion
The custom AdaBoost implementation successfully builds a strong classifier out of sequentially trained weak learners (Decision Stumps). 

By using weighted bootstrap sampling, our custom model forces each subsequent Decision Stump to focus on the misclassified samples from the previous iteration. The final prediction uses a weighted sum based on each stump's classification error rate ($\alpha$). 

Notice that the execution time is significantly faster than Random Forest. This is because AdaBoost restricts the depth of the tree to `max_depth=1`, which severely limits the recursive splitting process and computationally simplifies the algorithm.